## 09 - Feature requests

What do users ask the app to add? I find the reviews that ask for something new with three methods and check them against my own labels.

In [1]:
# --- IMPORTS ---
import json
import logging
import os
import re
import subprocess
import time
import warnings

# Three messages print local paths, so they are hidden before the imports that raise them:
# tqdm warns that ipywidgets is missing, openpyxl warns that it cannot read the Excel extension
# of the dropdown (the answers are read anyway), and torch logs a note about redirects on Windows.
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="Data Validation extension is not supported")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from openpyxl.styles import Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold

In [2]:
# --- SETTINGS ---
# Reviews drawn from each stratum (reviews that match the rules, and reviews that do not)
SAMPLE_PER_STRATUM = 200
# Fixed seed for the sample and its order, so the same sample comes out on every run
SAMPLE_SEED = 42
# The coding files. labeling/ is in .gitignore, because the files contain review texts.
SAMPLE_PATH = "../labeling/fr_sample.xlsx"
KEY_PATH = "../labeling/fr_key.csv"

In [3]:
# --- LOAD DATA ---
# The cleaned reviews, Italian only, as in 06. The rules read the original text, not text_clean:
# the cleaning of 03 removes apostrophes, which patterns such as "non c'è" and "l'opzione" need.
clean_df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")
italian_df = clean_df[clean_df["language"] == "it"].copy().reset_index(drop=True)

# Reviews with no text cannot match a rule and cannot be coded, so they are counted
n_empty = 0
for text in italian_df["text"]:
    if not isinstance(text, str) or len(text.strip()) == 0:
        n_empty += 1

print(f"Italian reviews        : {len(italian_df):,}")
print(f"Reviews with empty text: {n_empty}")

Italian reviews        : 6,951
Reviews with empty text: 0


## Part 1 - Rules, sample and coding file

The rules below are fixed before any labelling and are not changed after looking at my labels. If I changed them after seeing which reviews they miss, they would fit this sample and their measured precision and recall would be too optimistic. A review matches if its text contains at least one of the patterns.

In [4]:
# --- THE RULES ---
# Regular expressions searched in the lowercase text. \b is a word boundary, so "vorrei" does
# not match inside a longer word. Alternatives that end with an apostrophe have no \b after it.
# Accented letters also accept the grave accent, or an apostrophe in place of the accent,
# because both spellings are common: "perchè", "possibilita'", "c'e'", "puo'".
REQUEST_PATTERNS = [
    r"\bvorrei\b",
    r"\bvorremmo\b",
    r"\bmi piacerebbe\b",
    r"\bci vorrebbe\b",
    r"\bservirebbe\b",
    r"\bsarebbe (utile|bello|comodo|carino|ottimo|meglio|opportuno|il caso|da)\b",
    r"\bsi potrebbe (aggiungere|mettere|inserire|avere|fare)\b",
    r"\bpotreste (aggiungere|mettere|inserire|fare|dare)\b",
    r"\bdovreste (aggiungere|mettere|inserire|fare|dare)\b",
    r"\baggiungete\b",
    r"\binserite\b",
    r"\baggiungere (la\b|il\b|una\b|un\b|l')",
    r"\bperch[eèé]'? non (mettete|aggiungete|fate|inserite|si pu[oò]'?)\b",
    r"\bmanca (la possibilit[aà]'?|l'opzione|la funzione|una funzione|un'opzione)\b",
    r"\bnon c'[eèé]'? (la possibilit[aà]'?|l'opzione|la funzione|modo di)\b",
    r"\bpossibilit[aà]'? di\b",
    r"\bopzione per\b",
    r"\bsuggerisco\b",
    r"\bpropongo\b",
    r"\bconsiglio di (aggiungere|inserire|mettere)\b",
]

print(f"Patterns: {len(REQUEST_PATTERNS)}")

Patterns: 20


In [5]:
# --- PREPARE THE TEXTS FOR THE RULES ---
# A lowercase copy of each text, with the typographic apostrophes replaced by the plain one,
# so the rules see the same apostrophe whatever keyboard was used. The original text is not changed.
rule_texts = []
for text in italian_df["text"]:
    if not isinstance(text, str):
        text = ""
    text = text.lower()
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    rule_texts.append(text)

print(f"Texts prepared: {len(rule_texts):,}")

Texts prepared: 6,951


In [6]:
# --- APPLY THE RULES ---
# A review matches as soon as one pattern is found in it
rule_matches = []
for text in rule_texts:
    is_match = False
    for pattern in REQUEST_PATTERNS:
        if re.search(pattern, text):
            is_match = True
            break
    rule_matches.append(is_match)

italian_df["rule_match"] = rule_matches

n_match = italian_df["rule_match"].sum()
print(f"Reviews that match: {n_match:,} of {len(italian_df):,} ({n_match / len(italian_df) * 100:.1f}%)")

Reviews that match: 300 of 6,951 (4.3%)


In [7]:
# --- REVIEWS PER PATTERN ---
# How many reviews each pattern finds. A review can contain more than one pattern, so the
# counts add up to more than the reviews that match.
rows = []
for pattern in REQUEST_PATTERNS:
    n_reviews = 0
    for text in rule_texts:
        if re.search(pattern, text):
            n_reviews += 1
    rows.append({"pattern": pattern, "reviews": n_reviews})

pattern_counts = pd.DataFrame(rows).set_index("pattern")
pattern_counts

,reviews
pattern,
\bvorrei\b,35
\bvorremmo\b,0
\bmi piacerebbe\b,10
\bci vorrebbe\b,2
\bservirebbe\b,2
\bsarebbe (utile|bello|comodo|carino|ottimo|meglio|opportuno|il caso|da)\b,62
\bsi potrebbe (aggiungere|mettere|inserire|avere|fare)\b,1
\bpotreste (aggiungere|mettere|inserire|fare|dare)\b,2
\bdovreste (aggiungere|mettere|inserire|fare|dare)\b,4


In [8]:
# --- THE STRATA AND THEIR SIZES ---
# The sample is drawn separately from the reviews that match and from those that do not.
# These two sizes are the weights for the estimates on all reviews in the next parts.
n_no_match = len(italian_df) - n_match
stratum_table = pd.DataFrame({
    "stratum": ["match", "no match"],
    "reviews": [n_match, n_no_match],
    "share (%)": [round(n_match / len(italian_df) * 100, 1), round(n_no_match / len(italian_df) * 100, 1)],
}).set_index("stratum")
stratum_table

,reviews,share (%)
stratum,,
match,300,4.3
no match,6651,95.7


### The sample

I code 200 reviews that match and 200 that do not, instead of 400 drawn from all reviews. A plain random sample would contain few matching reviews, too few to measure how often the rules are right, while the non-matching half shows the requests the rules miss. Because the halves do not have the real proportions, estimates for all reviews weight each half by the size of its stratum.

In [9]:
# --- DRAW THE SAMPLE ---
# The same number of reviews from each stratum, drawn at random with a fixed seed.
# If fewer reviews match than the sample size, all of them are taken.
matching_df = italian_df[italian_df["rule_match"]]
non_matching_df = italian_df[~italian_df["rule_match"]]

n_from_matching = min(SAMPLE_PER_STRATUM, len(matching_df))
if n_from_matching < SAMPLE_PER_STRATUM:
    print(f"Only {len(matching_df)} reviews match: all of them are in the sample")

matching_sample = matching_df.sample(n=n_from_matching, random_state=SAMPLE_SEED)
non_matching_sample = non_matching_df.sample(n=SAMPLE_PER_STRATUM, random_state=SAMPLE_SEED)
sample_df = pd.concat([matching_sample, non_matching_sample])

print(f"Sampled from the reviews that match       : {len(matching_sample)}")
print(f"Sampled from the reviews that do not match: {len(non_matching_sample)}")
print(f"Reviews in the sample                     : {len(sample_df)}")
print(f"Distinct reviews                          : {sample_df['review_id'].nunique()}")

Sampled from the reviews that match       : 200
Sampled from the reviews that do not match: 200
Reviews in the sample                     : 400
Distinct reviews                          : 400


In [10]:
# --- SHUFFLE THE ORDER ---
# Without shuffling, the matching reviews would come first and the coder would know the rule
# result from the position. sample(frac=1) returns all rows in random order.
sample_df = sample_df.sample(frac=1, random_state=SAMPLE_SEED).reset_index(drop=True)

# The id is given after shuffling, so it says nothing about the rule result
sample_df["sample_id"] = range(1, len(sample_df) + 1)

# Longest run of reviews with the same rule result in a row, to check they are not in blocks
longest_run = 1
current_run = 1
sample_matches = sample_df["rule_match"].tolist()
for i in range(1, len(sample_matches)):
    if sample_matches[i] == sample_matches[i - 1]:
        current_run += 1
    else:
        current_run = 1
    if current_run > longest_run:
        longest_run = current_run

print(f"Longest run of the same rule result in a row: {longest_run}")

Longest run of the same rule result in a row: 16


In [11]:
# --- CODEBOOK ---
# What each answer means. It goes to its own sheet of the coding file.
CODEBOOK = {
    "yes": "the review asks for a feature or a change that the app does not offer today "
           "(for example invoices, purchase history, another payment method, refunds in the app, "
           "notifications for a line).",
    "no": "everything else, including reports that something does not work, generic requests to "
          "improve the app, missing stations or wrong data, praise, complaints about trains.",
    "mixed": "if a review reports a problem and also asks for something new, answer yes.",
    "what_is_requested": "a few words on what is asked, in Italian or English "
                         "(for example \"fattura\", \"storico acquisti\"). Leave empty when the answer is no.",
}

print(f"Codebook rows: {len(CODEBOOK)}")

Codebook rows: 4


In [12]:
# --- WRITE THE CODING FILES (only the first time) ---
# If the Excel file is already there, nothing is written, so a rerun never overwrites the coding.
if os.path.exists(SAMPLE_PATH):
    print("labeling/fr_sample.xlsx already exists: no file was written, the coding is kept")
else:
    os.makedirs("../labeling", exist_ok=True)

    # Sheet "coding": the review and two empty columns for my answer. The rule result is not here.
    coding_sheet = sample_df[["sample_id", "text"]].copy()
    coding_sheet["is_request"] = None
    coding_sheet["what_is_requested"] = None

    # Sheet "codebook": one row per entry with its definition
    codebook_rows = []
    for entry, definition in CODEBOOK.items():
        codebook_rows.append({"entry": entry, "definition": definition})
    codebook_sheet = pd.DataFrame(codebook_rows)

    with pd.ExcelWriter(SAMPLE_PATH, engine="openpyxl") as writer:
        coding_sheet.to_excel(writer, sheet_name="coding", index=False)
        codebook_sheet.to_excel(writer, sheet_name="codebook", index=False)

        # Dropdown with the two answers in the is_request column
        worksheet = writer.sheets["coding"]
        dropdown = DataValidation(type="list", formula1='"yes,no"', allow_blank=True, showErrorMessage=True)
        worksheet.add_data_validation(dropdown)
        dropdown.add(f"C2:C{len(coding_sheet) + 1}")

        # Wide, wrapped text column, so long reviews can be read without resizing
        worksheet.column_dimensions["B"].width = 100
        worksheet.column_dimensions["C"].width = 12
        worksheet.column_dimensions["D"].width = 40
        for row in worksheet.iter_rows(min_row=2, min_col=2, max_col=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

        codebook_worksheet = writer.sheets["codebook"]
        codebook_worksheet.column_dimensions["A"].width = 20
        codebook_worksheet.column_dimensions["B"].width = 100
        for row in codebook_worksheet.iter_rows(min_row=2, min_col=2, max_col=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

    # The hidden key: the rule result of each sample_id. It stays closed until the coding is done.
    key_df = sample_df[["sample_id", "review_id", "rule_match"]]
    key_df.to_csv(KEY_PATH, index=False, encoding="utf-8-sig")

    print("Created labeling/fr_sample.xlsx and labeling/fr_key.csv")

labeling/fr_sample.xlsx already exists: no file was written, the coding is kept


In [13]:
# --- CHECK THE CODING FILES ---
# The files are read back from disk. Only counts and True or False are printed, so the
# texts and the key stay hidden.
coding_back = pd.read_excel(SAMPLE_PATH, sheet_name="coding")
codebook_back = pd.read_excel(SAMPLE_PATH, sheet_name="codebook")
key_back = pd.read_csv(KEY_PATH)

# The coding sheet must have exactly these columns: no rule result in it
expected_columns = ["sample_id", "text", "is_request", "what_is_requested"]
right_columns = coding_back.columns.tolist() == expected_columns

# Same reviews in the same order as the sample drawn above. If the data has changed since
# the files were made, this is False and the files describe an older sample.
texts_in_file = coding_back["text"].fillna("").tolist()
texts_in_sample = sample_df["text"].fillna("").tolist()
same_texts = texts_in_file == texts_in_sample

# Same ids in the coding sheet and in the key, and the same reviews in the key as in the sample
same_ids = coding_back["sample_id"].tolist() == key_back["sample_id"].tolist()
same_reviews = key_back["review_id"].tolist() == sample_df["review_id"].tolist()

n_coded = coding_back["is_request"].notna().sum()

print(f"Rows in the coding sheet        : {len(coding_back)}")
print(f"Rows in the codebook sheet      : {len(codebook_back)}")
print(f"Rows in the key                 : {len(key_back)}")
print(f"Coding sheet without rule result: {right_columns}")
print(f"Same texts as the sample        : {same_texts}")
print(f"Same ids in sheet and key       : {same_ids}")
print(f"Same reviews in key and sample  : {same_reviews}")
print(f"Reviews coded so far            : {n_coded}")

Rows in the coding sheet        : 400
Rows in the codebook sheet      : 4
Rows in the key                 : 400
Coding sheet without rule result: True
Same texts as the sample        : True
Same ids in sheet and key       : True
Same reviews in key and sample  : True
Reviews coded so far            : 400


In [14]:
# --- CHECK THAT THE CODING FILES STAY OUT OF GIT ---
# git check-ignore exits with 0 when a path is ignored. git diff --cached lists the staged
# files, which must not include anything in labeling/.
sample_ignored = subprocess.run(["git", "check-ignore", "-q", SAMPLE_PATH]).returncode == 0
key_ignored = subprocess.run(["git", "check-ignore", "-q", KEY_PATH]).returncode == 0

staged = subprocess.run(["git", "diff", "--cached", "--name-only", "--", "../labeling"],
                        capture_output=True, text=True).stdout
nothing_staged = staged.strip() == ""

print(f"fr_sample.xlsx ignored by git : {sample_ignored}")
print(f"fr_key.csv ignored by git     : {key_ignored}")
print(f"Nothing in labeling/ is staged: {nothing_staged}")

assert sample_ignored and key_ignored and nothing_staged, "A coding file could end up in git"

fr_sample.xlsx ignored by git : True
fr_key.csv ignored by git     : True
Nothing in labeling/ is staged: True


## Part 2 - Three methods compared on my coding

I coded the 400 reviews by hand without seeing the rule result, and here three methods are compared with my labels: the rules of part 1, a classifier trained on embeddings, and an LLM. Every metric is weighted by the size of the strata, so it describes all 6,951 reviews. No method is applied to the other reviews yet.

In [15]:
# --- SETTINGS OF PART 2 ---
# Bootstrap: resamples drawn within each stratum, and their seed
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 42

# Classifier: the embedding model of 06, and the folds of the cross-validation with their seed
EMBEDDING_MODEL = "paraphrase-multilingual-mpnet-base-v2"
CV_FOLDS = 5
CV_SEED = 42

# LLM through OpenRouter. The dated model name fixes the exact version.
LLM_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "openai/gpt-5.4-mini-20260317"
# "none" is the lowest reasoning effort OpenRouter offers. A test call with a made-up review,
# before this run, showed that the model accepts it.
LLM_REASONING_EFFORT = "none"
# A failed call, or an answer that is not valid JSON, is tried again up to 3 more times
LLM_MAX_RETRIES = 3
# No new call is made once the cost reported by OpenRouter goes above this total
LLM_BUDGET_USD = 3.0
# The answers of the LLM, in labeling/ because they describe review texts
LLM_PATH = "../labeling/fr_llm_sample.csv"

In [16]:
# --- LOAD AND CHECK MY CODING ---
# Only my answers are read here. The key is read in the next cell, after the checks pass.
my_coding = pd.read_excel(SAMPLE_PATH, sheet_name="coding")

# Check 1: 400 rows, with the ids of the sample in the same order
right_rows = (len(my_coding) == len(sample_df)
              and my_coding["sample_id"].tolist() == sample_df["sample_id"].tolist())

# Check 2: every answer is "yes" or "no". An empty cell is read as NaN, so it counts as invalid.
n_invalid = 0
for answer in my_coding["is_request"]:
    if answer not in ["yes", "no"]:
        n_invalid += 1

# what_is_requested is not required. The descriptions that exist are kept but not used in any metric.
n_described = my_coding["what_is_requested"].notna().sum()

print(f"Rows, with the ids of the sample: {right_rows}")
print(f"Answers not yes or no, or empty : {n_invalid}")
print(f"Answers yes                     : {(my_coding['is_request'] == 'yes').sum()}")
print(f"Answers no                      : {(my_coding['is_request'] == 'no').sum()}")
print(f"Descriptions filled (not used)  : {n_described}")

assert right_rows and n_invalid == 0, "The coding is not complete: fix it before reading the key"

Rows, with the ids of the sample: True
Answers not yes or no, or empty : 0
Answers yes                     : 180
Answers no                      : 220
Descriptions filled (not used)  : 20


In [17]:
# --- JOIN MY CODING WITH THE KEY ---
# The key is read only now. merge() pairs each answer with the rule result of the same sample_id;
# validate="one_to_one" stops the notebook if an id appears twice in either file.
key_df = pd.read_csv(KEY_PATH)
coded_df = my_coding[["sample_id", "text", "is_request"]].merge(
    key_df, on="sample_id", how="inner", validate="one_to_one"
)

# my_yes is True where my answer is "yes"
coded_df["my_yes"] = coded_df["is_request"] == "yes"

# The joined table must hold the same reviews as the sample, in the same order
same_reviews = coded_df["review_id"].tolist() == sample_df["review_id"].tolist()
rule_is_boolean = coded_df["rule_match"].dtype == bool

print(f"Rows after the join            : {len(coded_df)}")
print(f"Same reviews as the sample     : {same_reviews}")
print(f"Rule result read as True/False : {rule_is_boolean}")
print(f"Sampled from the matching stratum    : {coded_df['rule_match'].sum()}")
print(f"Sampled from the non-matching stratum: {(~coded_df['rule_match']).sum()}")

assert len(coded_df) == len(my_coding) and same_reviews and rule_is_boolean, "The key does not fit my coding"

Rows after the join            : 400
Same reviews as the sample     : True
Rule result read as True/False : True
Sampled from the matching stratum    : 200
Sampled from the non-matching stratum: 200


### Weights

The sample has 200 reviews from each stratum, but the strata have 300 and 6,651 reviews. Each sampled review gets the weight N_s / 200, the number of reviews of its stratum it stands for, because matching reviews are 50% of the sample but only 4.3% of all reviews, and without weights the numbers would describe the sample instead of all 6,951 reviews.

In [18]:
# --- WEIGHTS ---
# N_s is the size of the stratum, from part 1; the denominator is the number of reviews
# sampled from it
stratum_size = {True: n_match, False: n_no_match}
sampled_per_stratum = {True: coded_df["rule_match"].sum(), False: (~coded_df["rule_match"]).sum()}

weights = []
for in_match in coded_df["rule_match"]:
    weights.append(stratum_size[in_match] / sampled_per_stratum[in_match])
coded_df["weight"] = weights

print(f"Weight of a matching review    : {stratum_size[True]} / {sampled_per_stratum[True]} = {stratum_size[True] / sampled_per_stratum[True]:.3f}")
print(f"Weight of a non-matching review: {stratum_size[False]} / {sampled_per_stratum[False]} = {stratum_size[False] / sampled_per_stratum[False]:.3f}")
print(f"Sum of the weights             : {coded_df['weight'].sum():,.0f} (all Italian reviews: {len(italian_df):,})")

Weight of a matching review    : 300 / 200 = 1.500
Weight of a non-matching review: 6651 / 200 = 33.255
Sum of the weights             : 6,951 (all Italian reviews: 6,951)


### Uncertainty

Every interval below comes from a stratified bootstrap: 2,000 times, 200 reviews are drawn with replacement within each stratum, and the number is computed again on the new sample. The same 2,000 resamples are used for every number, so the methods are compared on the same draws. The 95% interval goes from the 2.5th to the 97.5th percentile of the 2,000 values.

In [19]:
# --- BOOTSTRAP RESAMPLES ---
# Each resample keeps 200 reviews per stratum, drawn with replacement from that stratum, so the
# weights stay the same. Positions are row numbers of coded_df, which goes from 0 to 399.
rng = np.random.default_rng(BOOTSTRAP_SEED)
match_positions = np.where(coded_df["rule_match"])[0]
no_match_positions = np.where(~coded_df["rule_match"])[0]

resamples = []
for i in range(BOOTSTRAP_RESAMPLES):
    drawn_match = rng.choice(match_positions, size=len(match_positions), replace=True)
    drawn_no_match = rng.choice(no_match_positions, size=len(no_match_positions), replace=True)
    resamples.append(np.concatenate([drawn_match, drawn_no_match]))

# Plain arrays, used by every metric below
y_true = coded_df["my_yes"].to_numpy()
sample_weights = coded_df["weight"].to_numpy()

print(f"Resamples: {len(resamples):,}, each with {len(resamples[0])} reviews")

Resamples: 2,000, each with 400 reviews


In [20]:
# --- HOW COMMON REQUESTS ARE ---
# The share of my "yes" in each stratum, then the weighted estimate for all 6,951 reviews
rows = []
for in_match, name in [(True, "match"), (False, "no match")]:
    stratum_rows = coded_df[coded_df["rule_match"] == in_match]
    n_yes = stratum_rows["my_yes"].sum()
    rows.append({
        "stratum": name,
        "sampled": len(stratum_rows),
        "yes": n_yes,
        "yes (%)": round(n_yes / len(stratum_rows) * 100, 1),
        "stratum size": stratum_size[in_match],
        "weight": round(stratum_size[in_match] / len(stratum_rows), 3),
    })
request_table = pd.DataFrame(rows).set_index("stratum")
display(request_table)

# Each "yes" counts for as many reviews as its weight
estimated_requests = (sample_weights * y_true).sum()

boot_requests = []
for positions in resamples:
    boot_requests.append((sample_weights[positions] * y_true[positions]).sum())
requests_low, requests_high = np.percentile(boot_requests, [2.5, 97.5])

n_all = len(italian_df)
print(f"Estimated requests among all {n_all:,} reviews: {estimated_requests:,.0f} (95% interval {requests_low:,.0f} to {requests_high:,.0f})")
print(f"Estimated share of requests: {estimated_requests / n_all * 100:.1f}% (95% interval {requests_low / n_all * 100:.1f}% to {requests_high / n_all * 100:.1f}%)")

,sampled,yes,yes (%),stratum size,weight
stratum,,,,,
match,200,178,89.0,300,1.500
no match,200,2,1.0,6651,33.255


Estimated requests among all 6,951 reviews: 334 (95% interval 262 to 436)
Estimated share of requests: 4.8% (95% interval 3.8% to 6.3%)


### Method 1: rules

The rules of part 1, unchanged: a review is predicted as a request when it matches at least one pattern. The prediction is the rule result saved in the key.

In [21]:
# --- METHOD 1: RULES ---
rule_predictions = coded_df["rule_match"].to_numpy()

print(f"Reviews of the sample predicted as requests: {rule_predictions.sum()} of {len(rule_predictions)}")

Reviews of the sample predicted as requests: 200 of 400


### Method 2: classifier

Each review becomes a vector with the embedding model of 06, and a logistic regression with the scikit-learn defaults (max_iter=1000) learns to separate my yes from my no. The settings were fixed before seeing any result and are not tuned. With 5-fold cross-validation each review is predicted by a model trained on the other four fifths, so no review is predicted by a model that saw it.

In [22]:
# --- METHOD 2: EMBEDDINGS OF THE 400 TEXTS ---
# The original texts, the same that I read while coding
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
sample_texts = coded_df["text"].tolist()
sample_embeddings = embedding_model.encode(sample_texts, show_progress_bar=False)

print(f"Embeddings: {sample_embeddings.shape[0]} reviews x {sample_embeddings.shape[1]} dimensions")

Loading weights: 100%|██████████| 199/199 [00:04<00:00, 49.54it/s]


Embeddings: 400 reviews x 768 dimensions


In [23]:
# --- METHOD 2: CROSS-VALIDATED PREDICTIONS ---
# StratifiedKFold splits the 400 reviews into 5 folds with about the same share of "yes" in each.
# Each fold is predicted by a model trained on the other 4, so every review is predicted once.
folds = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=CV_SEED)

classifier_predictions = np.zeros(len(coded_df), dtype=bool)
times_predicted = np.zeros(len(coded_df), dtype=int)
fold_number = 0
for train_positions, test_positions in folds.split(sample_embeddings, y_true):
    fold_number += 1
    classifier = LogisticRegression(max_iter=1000)
    classifier.fit(sample_embeddings[train_positions], y_true[train_positions])
    classifier_predictions[test_positions] = classifier.predict(sample_embeddings[test_positions])
    times_predicted[test_positions] += 1
    print(f"Fold {fold_number}: trained on {len(train_positions)} reviews, predicted {len(test_positions)}")

assert (times_predicted == 1).all(), "A review was not predicted exactly once"
print(f"Reviews of the sample predicted as requests: {classifier_predictions.sum()} of {len(classifier_predictions)}")

Fold 1: trained on 320 reviews, predicted 80
Fold 2: trained on 320 reviews, predicted 80
Fold 3: trained on 320 reviews, predicted 80
Fold 4: trained on 320 reviews, predicted 80
Fold 5: trained on 320 reviews, predicted 80
Reviews of the sample predicted as requests: 169 of 400


### Method 3: LLM

An LLM reads each review through OpenRouter and answers with JSON, one call per review. It is zero-shot: the prompt gives only the codebook definitions, with no labelled example from the sample. The answers are saved, so a rerun only calls the API for the reviews still missing and gives the same results.

The prompt, sent as the system message; the review text alone is the user message.

```
You read one app review and decide whether it asks for something new.

Answer true if the review asks for a feature or a change that the app does not offer today (for example invoices, purchase history, another payment method, refunds in the app, notifications for a line).

Answer false for everything else, including reports that something does not work, generic requests to improve the app, missing stations or wrong data, praise, complaints about trains.

If a review reports a problem and also asks for something new, answer true.

Reply with JSON only, in this form: {"is_request": true or false, "what": "a few words on what is asked, in Italian or English, empty if false"}
```

In [24]:
# --- METHOD 3: PROMPT AND ANSWER FORMAT ---
# The same text as in the markdown cell above
LLM_PROMPT = """You read one app review and decide whether it asks for something new.

Answer true if the review asks for a feature or a change that the app does not offer today (for example invoices, purchase history, another payment method, refunds in the app, notifications for a line).

Answer false for everything else, including reports that something does not work, generic requests to improve the app, missing stations or wrong data, praise, complaints about trains.

If a review reports a problem and also asks for something new, answer true.

Reply with JSON only, in this form: {"is_request": true or false, "what": "a few words on what is asked, in Italian or English, empty if false"}"""

# The JSON schema of the answer: two fields, both required, nothing else allowed
LLM_SCHEMA = {
    "name": "feature_request",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "is_request": {"type": "boolean"},
            "what": {"type": "string"},
        },
        "required": ["is_request", "what"],
        "additionalProperties": False,
    },
}

print(f"Prompt: {len(LLM_PROMPT)} characters")

Prompt: 681 characters


In [25]:
# --- METHOD 3: CONNECT AND CHECK THE MODEL ---
# The key is read from .env and never printed
load_dotenv("../.env")
api_key = os.getenv("OPENROUTER_API_KEY")
assert api_key, "OPENROUTER_API_KEY is missing from .env"
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=api_key)

# The dated name is the canonical slug of a model in the OpenRouter list
llm_info = None
for model in llm_client.models.list().data:
    details = model.model_dump()
    if model.id == LLM_MODEL or details.get("canonical_slug") == LLM_MODEL:
        llm_info = details
assert llm_info is not None, f"{LLM_MODEL} is not on OpenRouter"

# Which settings the model accepts, from the same list
supported = llm_info["supported_parameters"]
use_temperature = "temperature" in supported
use_schema = "structured_outputs" in supported
if use_schema:
    response_format = {"type": "json_schema", "json_schema": LLM_SCHEMA}
else:
    response_format = {"type": "json_object"}

price_input = float(llm_info["pricing"]["prompt"]) * 1_000_000
price_output = float(llm_info["pricing"]["completion"]) * 1_000_000

print(f"Model found              : {LLM_MODEL}")
print(f"Listed as                : {llm_info['id']}, canonical slug {llm_info['canonical_slug']}")
print(f"Temperature accepted     : {use_temperature}")
print(f"Reasoning effort accepted: {'reasoning' in supported}, used: {LLM_REASONING_EFFORT}")
print(f"JSON schema accepted     : {use_schema}")
print(f"Price per million tokens : {price_input:.2f} USD input, {price_output:.2f} USD output")

Model found              : openai/gpt-5.4-mini-20260317
Listed as                : openai/gpt-5.4-mini, canonical slug openai/gpt-5.4-mini-20260317
Temperature accepted     : False
Reasoning effort accepted: True, used: none
JSON schema accepted     : True
Price per million tokens : 0.75 USD input, 4.50 USD output


In [26]:
# --- METHOD 3: FUNCTION FOR ONE CALL ---
def ask_llm(review_text):
    # One review, one call. Returns the answer, or None if it is not valid JSON with the two
    # fields, and the usage reported by OpenRouter. Errors of the API are raised to the caller.
    request = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": LLM_PROMPT},
            {"role": "user", "content": review_text},
        ],
        "response_format": response_format,
        # OpenRouter settings: the reasoning effort, and the usage with its cost in the response
        "extra_body": {"reasoning": {"effort": LLM_REASONING_EFFORT}, "usage": {"include": True}},
    }
    if use_temperature:
        request["temperature"] = 0

    response = llm_client.chat.completions.create(**request)

    usage = response.usage.model_dump()
    token_details = usage.get("completion_tokens_details") or {}
    usage_row = {
        "input_tokens": usage["prompt_tokens"],
        "output_tokens": usage["completion_tokens"],
        "reasoning_tokens": token_details.get("reasoning_tokens") or 0,
        "cost_usd": usage.get("cost") or 0.0,
    }

    content = response.choices[0].message.content or ""
    try:
        answer = json.loads(content)
    except ValueError:
        return None, usage_row

    is_valid = (isinstance(answer, dict)
                and isinstance(answer.get("is_request"), bool)
                and isinstance(answer.get("what"), str))
    if not is_valid:
        return None, usage_row
    return answer, usage_row

In [27]:
# --- METHOD 3: CALL THE LLM FOR THE REVIEWS NOT SAVED YET ---
# Answers saved by an earlier run are read first, and their cost counts towards the budget
saved_rows = []
if os.path.exists(LLM_PATH):
    saved_rows = pd.read_csv(LLM_PATH).to_dict("records")

saved_ids = set()
running_cost = 0.0
for row in saved_rows:
    saved_ids.add(row["sample_id"])
    running_cost += row["cost_usd"]
n_saved_before = len(saved_rows)

n_calls = 0
failed_ids = []
error_counts = {}
cost_of_failed_attempts = 0.0
budget_reached = False
for sample_id, review_text in zip(coded_df["sample_id"], coded_df["text"]):
    if sample_id in saved_ids:
        continue
    if running_cost > LLM_BUDGET_USD:
        budget_reached = True
        break

    answer = None
    for attempt in range(1 + LLM_MAX_RETRIES):
        n_calls += 1
        try:
            answer, usage_row = ask_llm(review_text)
            running_cost += usage_row["cost_usd"]
            if answer is None:
                cost_of_failed_attempts += usage_row["cost_usd"]
                error_counts["not valid JSON"] = error_counts.get("not valid JSON", 0) + 1
        except Exception as error:
            # Only the type of the error is kept: the message is not printed
            error_name = type(error).__name__
            error_counts[error_name] = error_counts.get(error_name, 0) + 1
            time.sleep(2 * (attempt + 1))
        if answer is not None:
            break

    if answer is None:
        failed_ids.append(int(sample_id))
        continue

    # Saved after every answer, so an interruption loses nothing
    new_row = {"sample_id": int(sample_id), "is_request": answer["is_request"], "what": answer["what"]}
    new_row.update(usage_row)
    saved_rows.append(new_row)
    pd.DataFrame(saved_rows).to_csv(LLM_PATH, index=False, encoding="utf-8-sig")

print(f"Answers saved before this run: {n_saved_before}")
print(f"API calls in this run        : {n_calls}")
print(f"Answers saved now            : {len(saved_rows)} of {len(coded_df)}")
print(f"Reviews still failing        : {len(failed_ids)} {failed_ids}")
print(f"Failed attempts by type      : {error_counts}")
print(f"Cost of failed attempts      : {cost_of_failed_attempts:.4f} USD")
print(f"Budget of {LLM_BUDGET_USD:.2f} USD reached: {budget_reached}")

Answers saved before this run: 0
API calls in this run        : 400
Answers saved now            : 400 of 400
Reviews still failing        : 0 []
Failed attempts by type      : {}
Cost of failed attempts      : 0.0000 USD
Budget of 3.00 USD reached: False


In [28]:
# --- METHOD 3: ANSWERS, TOKENS AND COST ---
# The answers are read back from disk. Only counts and totals are printed.
llm_df = pd.read_csv(LLM_PATH)
all_answered = sorted(llm_df["sample_id"].tolist()) == sorted(coded_df["sample_id"].tolist())

llm_ignored = subprocess.run(["git", "check-ignore", "-q", LLM_PATH]).returncode == 0

print(f"Answers in the file       : {len(llm_df)}")
print(f"Every review answered     : {all_answered}")
print(f"fr_llm_sample.csv ignored by git: {llm_ignored}")
print(f"Input tokens              : {llm_df['input_tokens'].sum():,}")
print(f"Output tokens             : {llm_df['output_tokens'].sum():,}")
print(f"  of which reasoning      : {llm_df['reasoning_tokens'].sum():,}")
print(f"Total cost of the answers : {llm_df['cost_usd'].sum():.4f} USD")

assert all_answered, "Some reviews have no LLM answer: the comparison cannot use all 400"
assert llm_ignored, "fr_llm_sample.csv could end up in git"

# The predictions, in the order of coded_df
llm_answer_by_id = dict(zip(llm_df["sample_id"], llm_df["is_request"]))
llm_predictions_list = []
for sample_id in coded_df["sample_id"]:
    llm_predictions_list.append(bool(llm_answer_by_id[sample_id]))
llm_predictions = np.array(llm_predictions_list)

print(f"Reviews of the sample predicted as requests: {llm_predictions.sum()} of {len(llm_predictions)}")

Answers in the file       : 400
Every review answered     : True
fr_llm_sample.csv ignored by git: True
Input tokens              : 90,897
Output tokens             : 8,779
  of which reasoning      : 0
Total cost of the answers : 0.1077 USD
Reviews of the sample predicted as requests: 154 of 400


### Comparison

Precision is the share of predicted requests that are requests in my coding, recall is the share of my requests that the method finds, and F1 is their harmonic mean. All three are weighted, so they describe all 6,951 reviews, and each has a 95% interval from the same bootstrap.

In [29]:
# --- FUNCTION: WEIGHTED PRECISION, RECALL AND F1 FOR "YES" ---
def weighted_metrics(true_labels, predicted_labels, row_weights):
    # zero_division=0 gives 0 instead of an error when a resample has no predicted or true "yes"
    precision = precision_score(true_labels, predicted_labels, sample_weight=row_weights, zero_division=0)
    recall = recall_score(true_labels, predicted_labels, sample_weight=row_weights, zero_division=0)
    f1 = f1_score(true_labels, predicted_labels, sample_weight=row_weights, zero_division=0)
    return precision, recall, f1

In [30]:
# --- COMPARISON TABLE ---
METHOD_PREDICTIONS = {
    "Rules": rule_predictions,
    "Classifier": classifier_predictions,
    "LLM": llm_predictions,
}

rows = []
for method, predictions in METHOD_PREDICTIONS.items():
    precision, recall, f1 = weighted_metrics(y_true, predictions, sample_weights)

    # The same metrics on every resample, for the intervals
    boot_precision = []
    boot_recall = []
    boot_f1 = []
    for positions in resamples:
        p, r, f = weighted_metrics(y_true[positions], predictions[positions], sample_weights[positions])
        boot_precision.append(p)
        boot_recall.append(r)
        boot_f1.append(f)

    rows.append({
        "method": method,
        "precision": round(precision, 3),
        "precision CI low": round(np.percentile(boot_precision, 2.5), 3),
        "precision CI high": round(np.percentile(boot_precision, 97.5), 3),
        "recall": round(recall, 3),
        "recall CI low": round(np.percentile(boot_recall, 2.5), 3),
        "recall CI high": round(np.percentile(boot_recall, 97.5), 3),
        "F1": round(f1, 3),
        "F1 CI low": round(np.percentile(boot_f1, 2.5), 3),
        "F1 CI high": round(np.percentile(boot_f1, 97.5), 3),
    })

comparison_table = pd.DataFrame(rows).set_index("method")
comparison_table

,precision,precision CI low,precision CI high,recall,recall CI low,recall CI high,F1,F1 CI low,F1 CI high
method,,,,,,,,,
Rules,0.890,0.845,0.930,0.801,0.612,1.000,0.843,0.720,0.956
Classifier,0.102,0.080,0.133,0.513,0.381,0.670,0.170,0.137,0.212
LLM,0.176,0.123,0.257,0.626,0.483,0.763,0.275,0.202,0.374


In [31]:
# --- WEIGHTED CONFUSION MATRICES ---
# Each cell is an estimated number of reviews among all 6,951: the weights add up to that total
for method, predictions in METHOD_PREDICTIONS.items():
    matrix = confusion_matrix(y_true, predictions, labels=[True, False], sample_weight=sample_weights)
    matrix_table = pd.DataFrame(
        matrix.round().astype(int),
        index=["my label: yes", "my label: no"],
        columns=["predicted yes", "predicted no"],
    )
    print(method)
    display(matrix_table)

Rules


,predicted yes,predicted no
my label: yes,267,67
my label: no,33,6584


Classifier


,predicted yes,predicted no
my label: yes,171,163
my label: no,1511,5106


LLM


,predicted yes,predicted no
my label: yes,209,125
my label: no,975,5643


### Reading the comparison

The rules are the best method: F1 0.843 (0.720 to 0.956), against 0.275 (0.202 to 0.374) for the LLM and 0.170 (0.137 to 0.212) for the classifier, and the intervals do not overlap, so the difference is larger than the uncertainty. The gap comes from precision, 0.890 for the rules against 0.176 for the LLM and 0.102 for the classifier, which flag as requests many reviews that are not (an estimated 975 and 1,511 of the 6,951). On recall the intervals overlap, from 0.612 to 1.000 for the rules, 0.483 to 0.763 for the LLM and 0.381 to 0.670 for the classifier, so which method finds more requests cannot be told with confidence. The recall of the rules rests on the 2 requests I found among the 200 non-matching reviews, which is why its interval is so wide.

In [32]:
# --- COST ESTIMATE FOR THE OTHER REVIEWS ---
# Matching reviews may be longer than the others, so the average cost is taken per stratum and
# multiplied by the reviews of that stratum that were not in the sample
llm_with_stratum = llm_df.merge(coded_df[["sample_id", "rule_match"]], on="sample_id")

n_other = len(italian_df) - len(coded_df)
estimated_cost = 0.0
for in_match, name in [(True, "match"), (False, "no match")]:
    stratum_answers = llm_with_stratum[llm_with_stratum["rule_match"] == in_match]
    average_cost = stratum_answers["cost_usd"].mean()
    average_tokens = (stratum_answers["input_tokens"] + stratum_answers["output_tokens"]).mean()
    n_not_sampled = stratum_size[in_match] - len(stratum_answers)
    estimated_cost += n_not_sampled * average_cost
    print(f"{name:<8}: {n_not_sampled:,} reviews not sampled, {average_tokens:.0f} tokens and {average_cost:.6f} USD per call on average")

print(f"Estimated cost for the other {n_other:,} reviews: {estimated_cost:.2f} USD")

match   : 100 reviews not sampled, 265 tokens and 0.000291 USD per call on average
no match: 6,451 reviews not sampled, 233 tokens and 0.000247 USD per call on average
Estimated cost for the other 6,551 reviews: 1.62 USD
